# Assignment 5: Distance and Centrality in Networks

# Starter Code


In [84]:
import networkx as nx

# load file
from google.colab import files
uploaded = files.upload()


Saving email-EuAll.txt to email-EuAll (1).txt


In [85]:
# Part 1: Load the complete directed network

def load_graph(file_path) -> nx.DiGraph:
  g = nx.read_edgelist(file_path, create_using=nx.DiGraph)
  return g

g = load_graph("email-EuAll.txt")


In [86]:
# Part 2: Largest strongly connected component

def largest_strongly_connected_subgraph(G: nx.DiGraph) -> nx.Graph:
  comp_list = sorted(nx.strongly_connected_components(G), key=len, reverse=True)[0]
  G_sc = G.subgraph(comp_list)
  return G_sc

In [87]:
# Part 3: Construct a manageable analysis graph

def create_analysis_graph(G_sc: nx.Graph, max_nodes=500) -> nx.Graph:
  G_un = G_sc.to_undirected()

  max_degree = max(dict(G_un.degree()).values())

  candidates = [node for node, degree in G_un.degree() if degree == max_degree]

  seed = sorted(candidates)[0]

  # Breadth-first traversal
  bfs_nodes = list(nx.bfs_tree(G_un, seed).nodes())

  selected_nodes = bfs_nodes[:max_nodes]

  H = G_un.subgraph(selected_nodes).copy()

  return H


In [88]:
# Part 4: Distance analysis

def distance_analysis(H: nx.Graph):

  eccent_values = nx.eccentricity(H)

  return {
      "avg_dist": nx.average_shortest_path_length(H),
      "diameter": nx.diameter(H, e=eccent_values),
      "radius": nx.radius(H, e=eccent_values),
      "center": nx.center(H, e=eccent_values),
      "periphery": nx.periphery(H, e=eccent_values),
      "eccentricity": eccent_values
  }

In [89]:
# Part 5: Centrality analysis

def centrality_analysis(H: nx.Graph):

  return {
      "deg_centrality": nx.degree_centrality(H),
      "close_centrality": nx.closeness_centrality(H),
      "between_centrality": nx.betweenness_centrality(H),
  }

In [90]:
# Main

file_path = "email-EuAll.txt"

G = load_graph(file_path)

print("Complete directed graph")
print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())

G_sc = largest_strongly_connected_subgraph(G)

print("\nLargest strongly connected component")
print("Nodes:", G_sc.number_of_nodes())
print("Edges:", G_sc.number_of_edges())

H = create_analysis_graph(G_sc, max_nodes=500)

print("\nAnalysis graph")
print("Nodes:", H.number_of_nodes())
print("Edges:", H.number_of_edges())
print("Connected:", nx.is_connected(H))

Complete directed graph
Nodes: 265214
Edges: 420045

Largest strongly connected component
Nodes: 34203
Edges: 151930

Analysis graph
Nodes: 500
Edges: 3384
Connected: True


# Part 1 - Load and Verify the Complete Network


In [91]:
print(f'Number of nodes: {g.number_of_nodes()}')
print(f'Number of edges: {g.number_of_edges()}')

Number of nodes: 265214
Number of edges: 420045


1. One directed edge represents a person sending at least one email to another person.

2. The original network is directed because it tracks emails between people.

3. It may be computationally expensive because of the amount of pairs that will exist since there are so many nodes and edges.

# Part 2 - Extract the Largest Strongly Connected Component

In [92]:
G_sc = largest_strongly_connected_subgraph(g)

print(f'Number of nodes: {G_sc.number_of_nodes()}')
print(f'Number of edges: {G_sc.number_of_edges()}')

Number of nodes: 34203
Number of edges: 151930


1. A strongly connected component is useful when studying distances in a directed graph because every distance will exist since every node is connected.

2. Calculating distances between nodes that cannot reach one another will cause the distance metrics to not be reliable since they will be calculating either wrong or missing distances.

# Part 3 - Construct the Analysis Graph

In [93]:
H = create_analysis_graph(G_sc)

print(f'Number of nodes: {H.number_of_nodes()}')
print(f'Number of edges: {H.number_of_edges()}')
print(f'Graph is connected: {nx.is_connected(H)}')

Number of nodes: 500
Number of edges: 3384
Graph is connected: True


# Part 4 - Shortest Paths and Average Distance

In [94]:
node1 = '16226'
node2 = '109154'

path_node1_node2 = nx.shortest_path(H, source=node1, target=node2)
length_path_node1_node2 = nx.shortest_path_length(H, source=node1, target=node2)

print(f'Shortest path between nodes {node1} and {node2}: {path_node1_node2}')
print(f'Distance between {node1} and {node2}: {length_path_node1_node2}')
print(f'Average shortest-path length: {distance_analysis(H)["avg_dist"]}')

Shortest path between nodes 16226 and 109154: ['16226', '192', '109154']
Distance between 16226 and 109154: 2
Average shortest-path length: 1.9737154308617235


1. The shortest-path distance between my two selected nodes is 2

2. This distance means that these two nodes are only separated by one node in the network.

3. The average shortest-path length of *H* is 1.974.

4. The average distance tells us that almost all of the nodes are separated by a distance of 2.

# Part 5 - Eccentricity

In [95]:
# used to list eccentricity values
# distance_analysis(H)[5]

node1 = '192'
node2 = '55034'

print(f'Low eccentricity: {nx.eccentricity(H, v=node1)}')
print(f'High eccentricity: {nx.eccentricity(H, v=node2)}')


Low eccentricity: 1
High eccentricity: 2


1. A small eccentricity means that this node is close to every node. In the case of node '192', it can reach every other node in a single hop.

2. A large eccentricity means that this node is not as close to every other node in the network.

3. Node '55034' is structurally closer to the rest of the network because the average distance is 1.974, which means that most nodes are not reachable in a single hop.

# Part 6 - Diameter, Radius, Center, and Periphery

In [96]:
dist_dict = distance_analysis(H)

print(f'Diameter: {dist_dict['diameter']}')
print(f'Radius: {dist_dict['radius']}')
print(f'Center: {dist_dict['center']}')
print(f'Periphery: {dist_dict['periphery']}')
len(dist_dict['periphery'])

Diameter: 2
Radius: 1
Center: ['192']
Periphery: ['7794', '94134', '5849', '20035', '35668', '14799', '26414', '26425', '18261', '14854', '1518', '16010', '107181', '15388', '3582', '13464', '1210', '104339', '13435', '1181', '2931', '24783', '58484', '939', '58550', '29890', '3681', '818', '83314', '406', '4066', '5353', '17151', '4324', '2797', '93199', '84590', '5864', '16563', '25881', '27495', '14432', '336', '27772', '84663', '8128', '5376', '24900', '84659', '104317', '80974', '54955', '1014', '19592', '13633', '920', '80656', '58763', '3370', '947', '5563', '28354', '26139', '11890', '55036', '16572', '2173', '65', '34014', '74542', '175', '4462', '195', '41385', '34626', '109', '516', '109955', '5866', '53472', '55124', '17116', '1844', '7296', '1992', '9747', '3933', '841', '18281', '4084', '28539', '3571', '1697', '73693', '21528', '28142', '38001', '15558', '64353', '5971', '389', '22372', '67078', '52560', '19182', '4701', '107754', '21706', '15365', '8013', '78991', '3507

499

1. The diameter of H is 2.

2. The radius is 1.

3. Node 192 belongs to the center.

4. Many nodes such as 7794, 94134, 5849, and 496 other nodes belong to the periphery.

5. A center and a peripheral node occupy different structural positions due to what is measured to find them. A center node is based off of the minimum eccentricity while a peripheral node is based off of the eccentricity equal to the diameter. One checks minimum eccentricity while the other checks eccentricity equal to the maximum distance between any pairs of node. They check two different positions within the network.

# Part 7 - Degree Centrality

In [97]:
cent_analysis = centrality_analysis(H)
highest_deg_cent = sorted(cent_analysis['deg_centrality'].items(), key=lambda x: x[1], reverse=True)[0:5]

print(f'Nodes: {highest_deg_cent}')
print('Node-Degree | Degree-Centrality')
for i in range(len(highest_deg_cent)):
  print(f'{H.degree(highest_deg_cent[i][0])} | {highest_deg_cent[i][1]}')

Nodes: [('192', 1.0040080160320641), ('336', 0.19839679358717432), ('406', 0.1903807615230461), ('1181', 0.1783567134268537), ('147', 0.17034068136272543)]
Node-Degree | Degree-Centrality
501 | 1.0040080160320641
99 | 0.19839679358717432
95 | 0.1903807615230461
89 | 0.1783567134268537
85 | 0.17034068136272543


| Node Degree |  Degree Centrality  |
|:-----------:|:-------------------:|
| 501         | 1.0040080160320641  |
| 99          | 0.19839679358717432 |
| 95          | 0.1903807615230461  |
| 89          | 0.1783567134268537  |
| 85          | 0.17034068136272543 |

1. Degree centrality measures the number of direct connections a node has, relative to the most it could have.

2. Degree centrality is related to degree but not the same numeric value because the degree centrality returns a fraction.

3. High degree-centrality usually means that the particular node is a hub or at least very influential.

# Part 8 - Closeness Centrality

In [98]:
close_cent = sorted(cent_analysis['close_centrality'].items(), key=lambda x: x[1], reverse=True)[0:5]

print('Node | Closeness Centrality')
for i in range(len(close_cent)):
  print(f'{close_cent[i][0]} | {close_cent[i][1]}')


Node | Closeness Centrality
192 | 1.0
336 | 0.5538290788013318
406 | 0.5513812154696133
1181 | 0.5477497255762898
147 | 0.5453551912568306


| Node | Closeness Centrality |
|:----:|:--------------------:|
| 192  | 1.0                  |
| 336  | 0.5538290788013318   |
| 406  | 0.5513812154696133   |
| 1181 | 0.5477497255762898   |
| 147  | 0.5453551912568306   |

1. Closeness centrality measures how close a node is to all other nodes on average.

2. Closeness centrality is connected to shortest-path distance in that it sums all of the shortest-path distances and averages them and then divided by the amount of nodes in a network - 1.

3. None of the center nodes made it in the highest closeness centrality.

4. No they are not the same nodes. This is due to closeness centrality and degree centrality measuring different metrics. Closeness is about how close a node is to all other nodes on average while degree centrality focuses on how many direct connections a node relative to the possible amount it can have.

# Part 9 - Betweeness Centrality

In [99]:
bet_cent = sorted(cent_analysis['between_centrality'].items(), key=lambda x: x[1], reverse=True)[0:5]

print('Node | Betweeness Centrality')
for i in range(len(bet_cent)):
  print(f'{bet_cent[i][0]} | {bet_cent[i][1]}')

Node | Betweeness Centrality
192 | 0.8426293871109138
79 | 0.008721148180127177
663 | 0.008666026789996721
1181 | 0.0075553741211723015
336 | 0.0071762274548369176


| Node | Betweenness Centrality |
|:----:|:----------------------:|
| 192  | 0.8426293871109138     |
| 79   | 0.008721148180127177   |
| 663  | 0.008666026789996721   |
| 1181 | 0.0075553741211723015  |
| 336  | 0.0071762274548369176  |

1. Betweenness centrality measures how often a node lies on the shortest paths between other pairs of nodes.

2. It means that the node is between the nodes and that in order to traverse from one node to another, that between node must be traveled through.

3. A node with a high betweenness centrality may be structurally important because many routes will have to pass through that node in order to make a traversal. It doesn't matter if that between node has a low degree because the degree only counts the immediate neighbors.

# Part 10 - Compare Centrality Measures




| Node |  Degree Centrality  | Closeness Centrality | Betweenness Centrality |
|:----:|:-------------------:|----------------------|-----------------------|
| 192  | 1.0040080160320641  | 1.0                  | 0.8426293871109138    |
| 336  | 0.19839679358717432 | 0.5538290788013318   | 0.0071762274548369176 |
| 406  | 0.1903807615230461  | 0.5513812154696133   |                       |
| 1181 | 0.1783567134268537  | 0.5477497255762898   | 0.0075553741211723015 |
| 147  | 0.17034068136272543 | 0.5453551912568306   |                       |

1. Yes, the same node happens to rank highly on all three measures.

2. Node 336 ranks highly on the degree and closeness centrality but its betwenness centrality is the lowest out of the five.

3. This can happen because these measure different things about the network. For example, node 336 ranks highly on two out of the three metrics which means that node 336 has a higher amount of direct connections than most other nodes and is closer to other nodes. The low betweenness centrality shows that node 336 is not a node that is required to reach others so it ends up with a lower score.

4. Degree centrality measures the direct local connections a node has. Closeness centrality checks the average distance a node has to the rest of the network so it measures the reach a node has. Betweenness centrality emphasizes connectivity because if a node with high betweenness centrality is removed, it may lead to networks being less connected.

# Part 11 - Connect Distance and Centrality

In [100]:
H.degree('192')

501

1. In network H, yes the center node also has high closeness centrality even though they both measure different things.

2. Yes, the node with the highest degree centrality has the lowest eccentricity in network H. Node 192 has a degree centrality o f 1.0 and the lowest eccentricity which is also one. This may not be the case every single time due to degree only counting direct neighbors while eccentricity dpeends on the distance.

3. Yes, in network H, the node 192 has the highest betweenness centrality and it also has the highest degree. Node 192 has a degree of 501 and a betweenness centrality of 0.843 which is the highest out of all the nodes.

4. Using multiple measures to understand a network can let us know about how the network is structured without explicitly graphing it. In the case of network H, we know that his network is very central due to node 192 having high values for every metric which means that node 192 is a hub.

# Part 12 - AI-Assisted Learning Documentation

1. How can I interpret closeness centrality? Provide some examples.

2. The easiest way to interpret closeness centrality is to convert it back into an average distance: since closeness = 1 / average distance, the average number of hops from the node to everyone else is 1 / closeness. Examples include email or social networks where a person with high closeness can get a message to everyone quickly and hears news early. A manager connected across departments would score higher than someone who only talks to their own team.

3. I verified that this was correct by cross referencing this information with the lecture notes and by looking up definitions and other examples online.

4. I did use this suggestion to help me build answers for some of the questions related to closeness centrality.

# Part 13 - Reflection

Different centrality measures are able to identify different nodes as structurally important due to what they measure. Degree centrality will measure how many nodes are directly connected to a specific node relative to the amount of connections possible. This helps us see if a node has a lot more neighbors than others. Closeness centrality measures how close a node is to all other nodes on average. This means that the node with the lowest distance between all other nodes will have a higher closeness centrality which could help us identify a node that could efficiently spread information to the network. Betweeness centrality measures whether a node lies on the shortest-paths between two other nodes. This helps us find the bridges in the network and could determine the flow of information between nodes. If a bridge is removed, new longer routes will be made or entire parts of the network may be cut off.